In [ ]:
!pip install bitsandbytes trl

In [ ]:
# 깃허브에서 compatibility_functions.py 파일을 다운로드합니다.
!wget https://raw.githubusercontent.com/rickiepark/fine-tuning-llm/refs/heads/main/compatibility_functions.py

In [3]:
import torch
from peft import prepare_model_for_kbit_training, get_peft_model, LoraConfig
from datasets import load_dataset, Dataset
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig, \
    DataCollatorForLanguageModeling, DataCollatorWithPadding, \
    DataCollatorWithFlattening, BitsAndBytesConfig
from trl.data_utils import pack_dataset
from trl.extras.dataset_formatting import FORMAT_MAPPING
#이제 trl에서 conversations_formatting_function()을 지원하지 않는다.
#그렇기에 이를 대체하기 위해 apply_chat_template()을 사용한다.
from trl.data_utils import apply_chat_template
from compatibility_functions import DataCollatorForCompletionOnlyLM

#tokenizer에 대해 알아보자.

In [4]:
#subword에 대한 tokenization
tokenizer=AutoTokenizer.from_pretrained("facebook/opt-350m")
quote="A noble spirit embiggens the smallest man."
print(tokenizer.tokenize(quote))
print(tokenizer.encode(quote,add_special_tokens=False))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/644 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/685 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/441 [00:00<?, ?B/s]

['A', 'Ġnoble', 'Ġspirit', 'Ġemb', 'igg', 'ens', 'Ġthe', 'Ġsmallest', 'Ġman', '.']
[250, 25097, 4780, 18484, 11702, 1290, 5, 15654, 313, 4]


In [5]:
#이전 장에서 quantization된 model에 LoRA adapter를 추가했다.
#이전 장에서 나온 코드

supported=torch.cuda.is_bf16_supported(including_emulation=False)
compute_dtype=(torch.bfloat16 if supported else torch.float32)

nf4_config=BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype
)

model_q4=AutoModelForCausalLM.from_pretrained(
    "facebook/opt-350m",device_map="cuda:0",dtype=compute_dtype,
    quantization_config=nf4_config
)

model_q4=prepare_model_for_kbit_training(model_q4)

config=LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

peft_model=get_peft_model(model_q4,config)

pytorch_model.bin:   0%|          | 0.00/663M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/388 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/662M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

In [6]:
repo_id="microsoft/phi-3-mini-4k-instruct"
tokenizer_phi=AutoTokenizer.from_pretrained(repo_id)
print(tokenizer_phi.chat_template)

config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

{% for message in messages %}{% if message['role'] == 'system' %}{{'<|system|>
' + message['content'] + '<|end|>
'}}{% elif message['role'] == 'user' %}{{'<|user|>
' + message['content'] + '<|end|>
'}}{% elif message['role'] == 'assistant' %}{{'<|assistant|>
' + message['content'] + '<|end|>
'}}{% endif %}{% endfor %}{% if add_generation_prompt %}{{ '<|assistant|>
' }}{% else %}{{ eos_token }}{% endif %}


In [7]:
messages=[
    {"role":"system","content":"You are a helpful AI assistant."},
    {"role":"user","content":"What is the capital of Argentina?"},
    {"role":"assistant","content":"Buenos Aires."}
]

formatted=tokenizer_phi.apply_chat_template(
    conversation=messages,tokenize=False,add_generation_prompt=False
)
print(formatted)

<|system|>
You are a helpful AI assistant.<|end|>
<|user|>
What is the capital of Argentina?<|end|>
<|assistant|>
Buenos Aires.<|end|>
<|endoftext|>


In [8]:
#model의 fine-tuning이 끝난 후 model을 trigger하고, prompt에 대한 대답을 생성하도록 해야한다.
#이것이 generation prompt를 추가해야 하는 이유이다.
inference_input=tokenizer_phi.apply_chat_template(
    conversation=messages[:-1], tokenize=False,add_generation_prompt=True
)
print(inference_input)

<|system|>
You are a helpful AI assistant.<|end|>
<|user|>
What is the capital of Argentina?<|end|>
<|assistant|>



In [9]:
#SFTTrainer class가 기본적으로 지원하는 대화 포맷
{"messasges":[
    {"role":"system","content":"<directives>"},
    {"role":"user","content":"<prompt>"},
    {"role":"assistant","content":"<completion>"}
]}

{'messasges': [{'role': 'system', 'content': '<directives>'},
  {'role': 'user', 'content': '<prompt>'},
  {'role': 'assistant', 'content': '<completion>'}]}

In [10]:
#메시지 데이터셋을 만든다.
conversation_ds=Dataset.from_list([{"messages":messages}])
conversation_ds.features


{'messages': List({'role': Value('string'), 'content': Value('string')})}

In [11]:
FORMAT_MAPPING["chatml"]==conversation_ds.features["messages"]

True

In [12]:
#conversations_formatting_function()은 삭제되었다.
"""
formatting_func=conversations_formatting_function(
  tokenizer_phi,messages_field="messages"
)
print(formatting_func(conversation_ds[0]))
"""

formatted = tokenizer_phi.apply_chat_template(conversation=conversation_ds[0]['messages'],
                                              tokenize=False,
                                              add_generation_prompt=False)
print(formatted)

<|system|>
You are a helpful AI assistant.<|end|>
<|user|>
What is the capital of Argentina?<|end|>
<|assistant|>
Buenos Aires.<|end|>
<|endoftext|>


In [13]:
#conversations_formatting_function() 함수의 코드는 다음과 같다.
def format_dataset(examples):
  if isinstance(examples[messages_field][0],list):
    output_texts=[]
    for i in range(len(examples[messages_field])):
      output_texts.append(tokenizer.apply_chat_template(
          examples[messages_field][i],tokenize=False
      ))
    return output_texts
  else:
    return tokenizer.apply_chat_template(examples[messages_field],
                                         tokenize=False)

In [14]:
#다음 함수로 지시 포맷으로 구성된 dataset을 대화 포맷으로 바꿀 수 있다.
def format_dataset(examples):
  if isinstance(examples["prompt"],list):
    output_texts=[]
    for i in range(len(examples["prompt"])):
      converted_sample=[
          {"role":"user","content":examples["prompt"][i]},
          {"role":"assistant","content":examples["completion"][i]},
      ]
      output_texts.append(converted_sample)
    return {"messages":output_texts}
  else:
    converted_sample=[
        {"role":"user","content":examples["prompt"]},
        {"role":"assistant","content":examples["completion"]},
    ]
  return {"messages":converted_sample}

In [15]:
#batch size가 2인 prompt와 완성 쌍이 있다
batch_prompts_completions={
    "prompt":["What is the capital of Argentina?",
              "What is the capital of the United States?"],
    "completion":["Buenos Aires.",
                  "Washington D.C."]
}

In [16]:
#위 데이터을 대화 포맷으로 바꾸겠다.
batch_messages=format_dataset(batch_prompts_completions)["messages"]
batch_messages

[[{'role': 'user', 'content': 'What is the capital of Argentina?'},
  {'role': 'assistant', 'content': 'Buenos Aires.'}],
 [{'role': 'user', 'content': 'What is the capital of the United States?'},
  {'role': 'assistant', 'content': 'Washington D.C.'}]]

#BYOFF(Bring Your Own Formatting Function)



In [17]:
def byo_formatting_func1(examples):
  messages=examples["messages"]
  output_texts=tokenizer_phi.apply_chat_template(
      messages,tokenize=False,add_generation_prompt=False
  )
  return output_texts

In [18]:
#위에서 직접 만든 formatting function을 테스트 해보겠다.
#tokenizer를 formatting된 batch 출력에 적용한다.
ds_msg=Dataset.from_dict({"messages":batch_messages})
ds_msg.map(lambda v: tokenizer_phi(byo_formatting_func1(v)),batched=True)

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

Dataset({
    features: ['messages', 'input_ids', 'attention_mask'],
    num_rows: 2
})

In [19]:
#일반적인 template을 사용하지 않고 직접 문자를 조립하기 시작하면 오류가 발생하기 쉬워진다.
#다음 함수는 사용자 콘텐츠와 어시스턴트 콘텐츠를 명확히 구분하기 때문에 오류가 발생하지는 않는다.
def byo_formatting_func2(examples):
  instruction_template="### Question:"
  response_template="### Answer"
  text=f"{instruction_template} {examples["prompt"]}\n"
  text+=f"{response_template} {examples["completion"]}"
  text+=tokenizer_phi.eos_token
  return text

In [20]:
ds_prompt=Dataset.from_dict(batch_prompts_completions)
print(byo_formatting_func2(ds_prompt[0]))

### Question: What is the capital of Argentina?
### Answer Buenos Aires.<|endoftext|>


In [21]:
#batch data를 사용해 위 함수 test
ds_prompt.map(lambda v: tokenizer_phi(byo_formatting_func2(v)),batched=True)

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

ArrowInvalid: Column 2 named input_ids expected length 2 but got length 43

In [22]:
#함수에 loop를 추가하고 추가된 loop를 사용하여 batch data를 처리한다.
def byo_formatting_func3(examples):
  output_texts=[]
  instruction_template="### Question:"
  response_template="### Answer:"
  for i in range(len(examples["prompt"])):
    text=f"{instruction_template} {examples["prompt"][i]}\n"
    text+=f"{response_template} {examples["completion"][i]}"
    text+=tokenizer_phi.eos_token
    output_texts.append(text)
  return output_texts

In [23]:
ds_prompt.map(lambda v: tokenizer_phi(byo_formatting_func3(v)),batched=True)

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

Dataset({
    features: ['prompt', 'completion', 'input_ids', 'attention_mask'],
    num_rows: 2
})

#BYOFD(Bring Your Own Formatted Data)

In [24]:
def byofd_formatting_func(examples):
  messages=examples["messages"]
  output_texts=tokenizer_phi.apply_chat_template(
      messages,tokenize=False,add_generation_prompt=False
  )
  return {"text":output_texts}

In [25]:
formatted_ds=ds_msg.map(byofd_formatting_func,batched=True)
formatted_ds["text"]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

Column(['<|user|>\nWhat is the capital of Argentina?<|end|>\n<|assistant|>\nBuenos Aires.<|end|>\n<|endoftext|>', '<|user|>\nWhat is the capital of the United States?<|end|>\n<|assistant|>\nWashington D.C.<|end|>\n<|endoftext|>'])

#tokenizer

In [26]:
repo_id="microsoft/phi-3-mini-4k-instruct"
tokenizer_phi=AutoTokenizer.from_pretrained(repo_id)
config_phi=AutoConfig.from_pretrained(repo_id)

In [27]:
tokenizer_phi("Let;s tokenize this sentence!")

{'input_ids': [2803, 29936, 29879, 5993, 675, 445, 10541, 29991], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1]}

In [28]:
#Phi-3의 tokenizer 차원과 모델 설정을 비교해보자.
#embedding layer의 차원이 vocabulary size보다 크다.
len(tokenizer_phi), config_phi.vocab_size

(32011, 32064)

In [29]:
sorted(tokenizer_phi.vocab.items(), key=lambda t: -t[1])[:11]

[('<|user|>', 32010),
 ('<|placeholder6|>', 32009),
 ('<|placeholder5|>', 32008),
 ('<|end|>', 32007),
 ('<|system|>', 32006),
 ('<|placeholder4|>', 32005),
 ('<|placeholder3|>', 32004),
 ('<|placeholder2|>', 32003),
 ('<|placeholder1|>', 32002),
 ('<|assistant|>', 32001),
 ('<|endoftext|>', 32000)]

In [30]:
tokenizer_phi.eos_token, tokenizer_phi.eos_token_id

('<|endoftext|>', 32000)

In [31]:
#Phi-3 model의 특수 token을 확인
tokenizer_phi.all_special_tokens

['<s>', '<|endoftext|>', '<unk>']

In [32]:
#<|endoftext|> token은 EOS와 PAD token의 역할을 모두 수행한다.
tokenizer_phi.special_tokens_map

{'bos_token': '<s>',
 'eos_token': '<|endoftext|>',
 'unk_token': '<unk>',
 'pad_token': '<|endoftext|>'}

In [33]:
#Phi-3에 분류 토큰, 분리 토큰, 마스크 토큰이 있는지 확인
(tokenizer_phi.cls_token, tokenizer_phi.sep_token,tokenizer_phi.mask_token)

(None, None, None)

In [34]:
#tokenizer에 새로운 token 추가
tokenizer_phi.add_special_tokens({
    "cls_token":"<cls>","sep_token":"<sep>","mask_token":"<mask>"
})
tokenizer_phi.special_tokens_map

{'bos_token': '<s>',
 'eos_token': '<|endoftext|>',
 'unk_token': '<unk>',
 'sep_token': '<sep>',
 'pad_token': '<|endoftext|>',
 'cls_token': '<cls>',
 'mask_token': '<mask>'}

In [35]:
#추가된 token은 vocabulary 끝에 추가된다.
sorted(tokenizer_phi.vocab.items(),key=lambda t: -t[1])[:14]

[('<mask>', 32013),
 ('<sep>', 32012),
 ('<cls>', 32011),
 ('<|user|>', 32010),
 ('<|placeholder6|>', 32009),
 ('<|placeholder5|>', 32008),
 ('<|end|>', 32007),
 ('<|system|>', 32006),
 ('<|placeholder4|>', 32005),
 ('<|placeholder3|>', 32004),
 ('<|placeholder2|>', 32003),
 ('<|placeholder1|>', 32002),
 ('<|assistant|>', 32001),
 ('<|endoftext|>', 32000)]

In [36]:
#padding을 위해 사용할 token 설정(unk token을 pad token으로 같이 사용)
tokenizer_phi.pad_token=tokenizer_phi.unk_token
tokenizer_phi.pad_token_id=tokenizer_phi.unk_token_id

tokenizer_phi.special_tokens_map

{'bos_token': '<s>',
 'eos_token': '<|endoftext|>',
 'unk_token': '<unk>',
 'sep_token': '<sep>',
 'pad_token': '<unk>',
 'cls_token': '<cls>',
 'mask_token': '<mask>'}

In [37]:
#tokenizer에 있는 PAD, BOS, EOS token을 수정하면 model 설정에서 해당 tokenID도 수정해야 한다.
#수정된 padding token을 위해 model 설정을 수정
if getattr(model_q4,"config",None) is not None:
  model_q4.config.pad_token_id=tokenizer_phi.pad_token_id
if (getattr(model_q4, "generation_config",None) is not None):
  model_q4.config.pad_token_id=tokenizer_phi.pad_token_id

In [38]:
tokenizer_phi.pad_token_id, tokenizer_phi.unk_token_id, model_q4.config.pad_token_id

(0, 0, 0)

In [39]:
#Phi-3의 padding은 다음과 같이 설정되어 있다.
tokenizer_phi.pad_token, tokenizer_phi.padding_side

('<unk>', 'left')

#data collator

여러 개의 datasample을 하나의 mini-batch로 묶는 역할을 한다.

In [40]:
#yoda_sentences dataset load
dataset=load_dataset("dvgodoy/yoda_sentences",split="train")
dataset=dataset.rename_column("sentence","prompt")
dataset=dataset.rename_column("translation_extra","completion")
#prompt/완성 쌍을 대화 메시지로 변환한다.
dataset=dataset.map(format_dataset)
dataset=dataset.remove_columns(["prompt","completion","translation"])
len(dataset),dataset[0]

README.md:   0%|          | 0.00/531 [00:00<?, ?B/s]

sentences.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/720 [00:00<?, ? examples/s]

Map:   0%|          | 0/720 [00:00<?, ? examples/s]

(720,
 {'messages': [{'role': 'user',
    'content': 'The birch canoe slid on the smooth planks.'},
   {'role': 'assistant',
    'content': 'On the smooth planks, the birch canoe slid. Yes, hrrrm.'}]})

In [41]:
#formatting function을 전체 dataset에 적용한다.
def formatting_func(row):
  return tokenizer_phi.apply_chat_template(row["messages"],tokenize=False,
                                           add_generation_prompt=False)
dataset=dataset.map(lambda row:{"text":formatting_func(row)},batched=True,batch_size=32)
sequences=dataset["text"]
print(sequences[:2])

Map:   0%|          | 0/720 [00:00<?, ? examples/s]

['<|user|>\nThe birch canoe slid on the smooth planks.<|end|>\n<|assistant|>\nOn the smooth planks, the birch canoe slid. Yes, hrrrm.<|end|>\n<|endoftext|>', '<|user|>\nGlue the sheet to the dark blue background.<|end|>\n<|assistant|>\nGlue the sheet to the dark blue background, you must.<|end|>\n<|endoftext|>']


In [42]:
#dataset을 tokenization하고, tokenID만 남긴다.
tokenized_dataset=dataset.map(lambda row: tokenizer_phi(row["text"]))
tokenized_dataset=tokenized_dataset.select_columns(["input_ids"])

Map:   0%|          | 0/720 [00:00<?, ? examples/s]

In [43]:
pad_collator=DataCollatorWithPadding(tokenizer_phi)
pad_dloader=DataLoader(tokenized_dataset,batch_size=2,collate_fn=pad_collator)
pad_batch=next(iter(pad_dloader))
pad_batch

{'input_ids': tensor([[32010,   450, 29773,   305,   508,  7297,  2243,   333,   373,   278,
         10597,   715,  1331, 29889, 32007, 32001,  1551,   278, 10597,   715,
          1331, 29892,   278, 29773,   305,   508,  7297,  2243,   333, 29889,
          3869, 29892,   298, 21478,  1758, 29889, 32007, 32000],
        [    0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
         32010,  8467,   434,   278,  9869,   304,   278,  6501,  7254,  3239,
         29889, 32007, 32001,  8467,   434,   278,  9869,   304,   278,  6501,
          7254,  3239, 29892,   366,  1818, 29889, 32007, 32000]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [44]:
#dataset을 packing하지 않고, padding
lm_collator=DataCollatorForLanguageModeling(tokenizer_phi,mlm=False)
lm_dloader=DataLoader(tokenized_dataset,batch_size=2,collate_fn=lm_collator)
lm_batch=next(iter(lm_dloader))
lm_batch

{'input_ids': tensor([[32010,   450, 29773,   305,   508,  7297,  2243,   333,   373,   278,
         10597,   715,  1331, 29889, 32007, 32001,  1551,   278, 10597,   715,
          1331, 29892,   278, 29773,   305,   508,  7297,  2243,   333, 29889,
          3869, 29892,   298, 21478,  1758, 29889, 32007, 32000],
        [    0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
         32010,  8467,   434,   278,  9869,   304,   278,  6501,  7254,  3239,
         29889, 32007, 32001,  8467,   434,   278,  9869,   304,   278,  6501,
          7254,  3239, 29892,   366,  1818, 29889, 32007, 32000]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]), 'labels': tensor([[32010,   450, 29773,   305,   508,  7297,  2243,   333,   373,   278,
   

In [45]:
response_template="<|assistant|>" #tokenID 32001
completion_collator=DataCollatorForCompletionOnlyLM(
    response_template=response_template,tokenizer=tokenizer_phi
)
completion_dloader=DataLoader(
    tokenized_dataset, batch_size=2, collate_fn=completion_collator
)
completion_batch=next(iter(completion_dloader))
completion_batch

{'input_ids': tensor([[32010,   450, 29773,   305,   508,  7297,  2243,   333,   373,   278,
         10597,   715,  1331, 29889, 32007, 32001,  1551,   278, 10597,   715,
          1331, 29892,   278, 29773,   305,   508,  7297,  2243,   333, 29889,
          3869, 29892,   298, 21478,  1758, 29889, 32007, 32000],
        [    0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
         32010,  8467,   434,   278,  9869,   304,   278,  6501,  7254,  3239,
         29889, 32007, 32001,  8467,   434,   278,  9869,   304,   278,  6501,
          7254,  3239, 29892,   366,  1818, 29889, 32007, 32000]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]), 'labels': tensor([[ -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
   

In [46]:
labels=completion_batch["labels"][0]
valid_tokens=(labels>=0)
tokenizer_phi.decode(labels[valid_tokens])

'On the smooth planks, the birch canoe slid. Yes, hrrrm.<|end|><|endoftext|>'

In [47]:
#user와 AI assistant 간 chatting 이력이 있다고 가정
dummy_chat="""<|user|>Hello
<|assistant|>How are you?
<|user|>I'm fine! You?
<|assistant|>I'm fine too!
<|endoftext|>"""

dummy_ds=Dataset.from_dict({"text":[dummy_chat]})
dummy_ds=(dummy_ds.map(
    lambda row: tokenizer_phi(row["text"])).select_columns(["input_ids"])
)

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

In [48]:
completion_dloader=DataLoader(dummy_ds,batch_size=1,collate_fn=completion_collator)
completion_batch=next(iter(completion_dloader))
completion_batch

{'input_ids': tensor([[32010, 15043,    13, 32001,  1128,   526,   366, 29973,    13, 32010,
           306, 29915, 29885,  2691, 29991,   887, 29973,    13, 32001,   306,
         29915, 29885,  2691,  2086, 29991,    13, 32000]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1]]), 'labels': tensor([[ -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
          -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,   306,
         29915, 29885,  2691,  2086, 29991,    13, 32000]])}

In [50]:
labels=completion_batch["labels"]
tokenizer_phi.decode(labels[labels>=0])

"I'm fine too!\n<|endoftext|>"

In [51]:
instruction_template="<|user|>"
response_template="<|assistant|>"
completion_collator=DataCollatorForCompletionOnlyLM(
    instruction_template=instruction_template,response_template=response_template,
    tokenizer=tokenizer_phi
)
completion_dloader=DataLoader(dummy_ds,batch_size=1,collate_fn=completion_collator)
completion_batch=next(iter(completion_dloader))
completion_batch

{'input_ids': tensor([[32010, 15043,    13, 32001,  1128,   526,   366, 29973,    13, 32010,
           306, 29915, 29885,  2691, 29991,   887, 29973,    13, 32001,   306,
         29915, 29885,  2691,  2086, 29991,    13, 32000]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1]]), 'labels': tensor([[ -100,  -100,  -100,  -100,  1128,   526,   366, 29973,    13,  -100,
          -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,   306,
         29915, 29885,  2691,  2086, 29991,    13, 32000]])}

In [52]:
labels=completion_batch["labels"]
tokenizer_phi.decode(labels[labels>=0])

"How are you?\n I'm fine too!\n<|endoftext|>"

In [ ]:
#model이 훈련 과정에서 loss을 어떻게 계산하는지 알려주는 코드
if labels is not None:
  #model 병렬화를 위해 label을 올바른 장치로 이동시킨다.
  labels=labels.to(lm_logits.device)
  #next token predict을 수행하므로 예측 점수와 입력 ID을 하나씩 이동시킨다.
  shift_logits=lm_logits[:,:-1,:].contiguous()
  labels=labels[:,1:].contiguous()
  loss_fct=CrossEntropyLoss()
  lm_loss=loss_fct(shift_logits.vew(-1,shift_logits.size(-1)),labels.view(-1))

In [55]:
sequences=dataset["text"]
print(sequences[:2])

['<|user|>\nThe birch canoe slid on the smooth planks.<|end|>\n<|assistant|>\nOn the smooth planks, the birch canoe slid. Yes, hrrrm.<|end|>\n<|endoftext|>', '<|user|>\nGlue the sheet to the dark blue background.<|end|>\n<|assistant|>\nGlue the sheet to the dark blue background, you must.<|end|>\n<|endoftext|>']


In [56]:
#dataset을 packing하면 '\n'은 삭제하고 모든 sequence을 연결한다.
#그 다음 동일 크기의 chunk로 분할한다.

#dataset에 있는 sequence를 seq_length크기의 chunk로 packing
packed_dataset=pack_dataset(tokenized_dataset,seq_length=64,strategy="wrapped")

Map:   0%|          | 0/720 [00:00<?, ? examples/s]

In [57]:
packed_dataset

Dataset({
    features: ['input_ids'],
    num_rows: 341
})

In [58]:
input_ids=packed_dataset["input_ids"]
tokenizer_phi.decode(input_ids[0])

'<|user|> The birch canoe slid on the smooth planks.<|end|><|assistant|> On the smooth planks, the birch canoe slid. Yes, hrrrm.<|end|><|endoftext|><|user|> Glue the sheet to the dark blue background.<|end|><|assistant|> Glue the sheet to the dark blue background, you must.'

In [59]:
flat_collator=DataCollatorWithFlattening()
flat_dloader=DataLoader(tokenized_dataset,batch_size=2,
                        collate_fn=flat_collator)
flat_batch=next(iter(flat_dloader))
flat_batch

{'input_ids': tensor([[32010,   450, 29773,   305,   508,  7297,  2243,   333,   373,   278,
          10597,   715,  1331, 29889, 32007, 32001,  1551,   278, 10597,   715,
           1331, 29892,   278, 29773,   305,   508,  7297,  2243,   333, 29889,
           3869, 29892,   298, 21478,  1758, 29889, 32007, 32000, 32010,  8467,
            434,   278,  9869,   304,   278,  6501,  7254,  3239, 29889, 32007,
          32001,  8467,   434,   278,  9869,   304,   278,  6501,  7254,  3239,
          29892,   366,  1818, 29889, 32007, 32000]]),
 'labels': tensor([[ -100,   450, 29773,   305,   508,  7297,  2243,   333,   373,   278,
          10597,   715,  1331, 29889, 32007, 32001,  1551,   278, 10597,   715,
           1331, 29892,   278, 29773,   305,   508,  7297,  2243,   333, 29889,
           3869, 29892,   298, 21478,  1758, 29889, 32007, 32000,  -100,  8467,
            434,   278,  9869,   304,   278,  6501,  7254,  3239, 29889, 32007,
          32001,  8467,   434,   278,  986

In [60]:
flat_batch["input_ids"].shape, flat_batch["position_ids"].max() + 1

(torch.Size([1, 66]), tensor(38))

In [61]:
response_template="<|assistant|>"
completion_nopad_collator=DataCollatorForCompletionOnlyLM(
    response_template=response_template,tokenizer=tokenizer_phi,
    padding_free=True
)
completion_nopad_dloader=DataLoader(
    tokenized_dataset,batch_size=2,collate_fn=completion_nopad_collator
)
completion_nopad_batch=next(iter(completion_nopad_dloader))
completion_nopad_batch


{'input_ids': tensor([[32010,   450, 29773,   305,   508,  7297,  2243,   333,   373,   278,
         10597,   715,  1331, 29889, 32007, 32001,  1551,   278, 10597,   715,
          1331, 29892,   278, 29773,   305,   508,  7297,  2243,   333, 29889,
          3869, 29892,   298, 21478,  1758, 29889, 32007, 32000, 32010,  8467,
           434,   278,  9869,   304,   278,  6501,  7254,  3239, 29889, 32007,
         32001,  8467,   434,   278,  9869,   304,   278,  6501,  7254,  3239,
         29892,   366,  1818, 29889, 32007, 32000]]), 'labels': tensor([[ -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
          -100,  -100,  -100,  -100,  -100,  -100,  1551,   278, 10597,   715,
          1331, 29892,   278, 29773,   305,   508,  7297,  2243,   333, 29889,
          3869, 29892,   298, 21478,  1758, 29889, 32007, 32000,  -100,  -100,
          -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
          -100,  8467,   434,   278,  9869,   304,   

#BYOT(Bring Your Own Template)

In [63]:
repo_id="facebook/opt-350m"
model_opt=AutoModelForCausalLM.from_pretrained(repo_id)
tokenizer_opt=AutoTokenizer.from_pretrained(repo_id)
print(tokenizer_opt.chat_template)  #model opt에는 정의된 template이 없다고 출력된다.

Loading weights:   0%|          | 0/388 [00:00<?, ?it/s]

None


In [64]:
#사용 가능한 특수 token확인
tokenizer_opt.special_tokens_map

{'bos_token': '</s>',
 'eos_token': '</s>',
 'unk_token': '</s>',
 'pad_token': '<pad>'}

In [71]:
def get_multiple_of(vocab_size):
  return 2**(bin(vocab_size)[::-1].find("1"))

pad_to_multiple_of=get_multiple_of(model_opt.config.vocab_size)
pad_to_multiple_of

32

In [74]:
new_num_tokens=len(tokenizer_opt)
len(tokenizer_opt)

50265

In [73]:
new_num_tokens=((new_num_tokens + pad_to_multiple_of - 1)//pad_to_multiple_of)*pad_to_multiple_of

In [75]:
new_num_tokens

50265

In [76]:
model_opt.resize_token_embeddings(
    len(tokenizer_opt), pad_to_multiple_of=pad_to_multiple_of
)

Embedding(50272, 512, padding_idx=1)

In [83]:
#tokenizer 수정하기
def modify_tokenizer(tokenizer,
                     alternative_bos_token="<|im_start|>",
                     alternative_unk_token="<unk>",
                     special_tokens=None,
                     tokens=None):
  eos_token,bos_token=tokenizer.eos_token,tokenizer.bos_token
  pad_token,unk_token=tokenizer.pad_token,tokenizer.unk_token
  #BOS token은 EOS token과 달라야 한다.
  if bos_token==eos_token:
    bos_token=alternative_bos_token
  #UNK token은 EOS token과 달라야 한다.
  if unk_token==eos_token:
    unk_token=alternative_unk_token
  #PAD token은 EOS token과 달라야 한다.
  #하지만 UNK token과는 같을 수 있다.
  if pad_token==eos_token:
    pad_token=unk_token

  assert bos_token!=eos_token, "다른 BOS token을 선택하세요."
  assert unk_token!=eos_token, "다른 UNK token을 선택하세요."

  #BOS, PAD, UNK token을 위한 dictionary를 만든다.
  #EOS token은 원래 저으이된 대로 유지한다.
  special_tokens_dict={
      "bos_token":bos_token, "pad_token":pad_token,"unk_token":unk_token
  }

  #새로운 특수 token 추가
  if special_tokens is not None:
    if isinstance(special_tokens, list):
      special_tokens_dict.update({"additional_special_tokens":special_tokens})
  tokenizer.add_special_tokens(special_tokens_dict)
  #새로운 일반 token을 추가한다.
  if tokens is not None:
    if isinstance(tokens,list):
      tokenizer.add_tokens(tokens)

  return tokenizer

In [127]:
#chatting template 생성 함수
def jinja_template(tokenizer):
  return ("{% for message in messages %}"
          f"{{{{'{tokenizer.bos_token}'+ message['role'] + '\n' \
          + message['content'] + '{tokenizer.eos_token}' +'\n'}}}}"
          "{% endfor %}"
          "{% if add_generation_prompt %}"
          f"{{{{ '{tokenizer.bos_token}assistant\n' }}}}"
          "{% endif %}")
def add_template(tokenizer,chat_template=None):
  #chatting template이 주어지지 않으면 BOS와 EOS token을 사용하여, chatting template을 만든다.
  if chat_template is None:
      chat_template=jinja_template(tokenizer)

  #chatting template을 tokenizer에 할당한다.
  tokenizer.chat_template=chat_template
  return tokenizer


In [81]:
def get_multiple_of(vocab_size):
  return 2**(bin(vocab_size)[::-1].find('1'))

def modify_model(model,tokenizer):
  #새로운 tokenzier의 크기가 vocabulary 크기를 초과한다면, 같은 배수가 되도록 유지하면서 크기를 바꾼다.
  if len(tokenizer)>model.config.vocab_size:
    pad_to_multiple_of=get_multiple_of(model.vocab_size)
    model.resize_token_embeddings(
        len(tokenzier),pad_to_multiple_of=pad_to_multiple_of
    )
  #model 설정의 tokenID을 업데이트한다.
  if getattr(model, "config",None) is not None:
    model.config.pad_token_id=tokenizer.pad_token_id
    model.config.bos_token_id=tokenizer.bos_token_id
    model.config.eos_token_id=tokenizer.eos_token_id
  if getattr(model, "generation_config",None) is not None:
    model.generation_config.bos_token_id=tokenizer.bos_token_id
    model.generation_config.eos_token_id=tokenizer.eos_token_id
    model.generation_config.pad_token_id=tokenizer.pad_token_id

  return model

In [128]:
tokenizer_opt=modify_tokenizer(tokenizer_opt)
tokenizer_opt=add_template(tokenizer_opt)
model_opt=modify_model(model_opt,tokenizer_opt)

In [95]:
tokenizer_opt.special_tokens_map

{'bos_token': '<|im_start|>',
 'eos_token': '</s>',
 'unk_token': '<unk>',
 'pad_token': '<pad>'}

In [96]:
len(tokenizer_opt)

50266

In [97]:
tokenizer_opt.convert_ids_to_tokens(50265)

'<|im_start|>'

In [98]:
model_opt.get_input_embeddings()

Embedding(50272, 512, padding_idx=1)

In [129]:
print(tokenizer_opt.chat_template)

{% for message in messages %}{{'<|im_start|>'+ message['role'] + '
'           + message['content'] + '</s>' +'
'}}{% endfor %}{% if add_generation_prompt %}{{ '<|im_start|>assistant
' }}{% endif %}


In [130]:
messages=ds_msg["messages"][0]
print(tokenizer_opt.apply_chat_template(messages,tokenize=False))

<|im_start|>user
What is the capital of Argentina?</s>
<|im_start|>assistant
Buenos Aires.</s>



In [131]:
tokenizer_opt.model_max_length

1000000000000000019884624838656

In [135]:
tokenizer_opt.model_max_length=min(tokenizer_opt.model_max_length, model_opt.config.max_position_embeddings)

In [136]:
tokenizer_opt.model_max_length

2048

In [139]:
repo_id="facebook/opt-350m"
model_opt=AutoModelForCausalLM.from_pretrained(repo_id)
tokenizer_opt=AutoTokenizer.from_pretrained(repo_id)

response_template="##[YODA]##>"
tokenzier_opt=modify_tokenizer(tokenizer_opt,
                               special_tokens=[response_template])
model_opt=modify_model(model_opt,tokenizer_opt)

Loading weights:   0%|          | 0/388 [00:00<?, ?it/s]

In [140]:
#formatting 함수
def formatting_func_builder(response_template):
  def formatting_func(examples, add_generation_prompt=False):
    output_texts=[]
    for i in range(len(examples["prompt"])):
      text=f"{examples["prompt"][i]}"
      try:
        text+=f" {response_template}"
        text+=f" {examples["completion"][i]}{tokenizer_opt.eos_token}"
      except KeyError:
        if add_generation_prompt:
          text+=f" {response_template}"
      output_texts.append(text)
    return output_texts
  return formatting_func

yoda_formatting_func=formatting_func_builder(response_template)
yoda_formatting_func

<function __main__.formatting_func_builder.<locals>.formatting_func(examples, add_generation_prompt=False)>

In [141]:
#위 함수로 data formatting해보겠다.
dataset=load_dataset("dvgodoy/yoda_sentences",split="train")
dataset=dataset.rename_column("sentence","prompt")
dataset=dataset.rename_column("translation_extra","completion")

formatted_seqs=yoda_formatting_func(dataset)
formatted_seqs[0]

'The birch canoe slid on the smooth planks. ##[YODA]##> On the smooth planks, the birch canoe slid. Yes, hrrrm.</s>'

In [142]:
#tokenization
tokenizer_opt(formatted_seqs[0])

{'input_ids': [2, 133, 23629, 611, 31728, 13763, 15, 5, 6921, 563, 2258, 4, 1437, 50266, 374, 5, 6921, 563, 2258, 6, 5, 23629, 611, 31728, 13763, 4, 3216, 6, 1368, 28015, 22900, 4, 2], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [143]:
tokenizer_opt.convert_ids_to_tokens(50266)

'##[YODA]##>'

In [144]:
yoda_formatting_func({"prompt":["The Force is strong in you.",
                                "I am your father!"]},add_generation_prompt=True)

['The Force is strong in you. ##[YODA]##> ##[YODA]##>',
 'I am your father! ##[YODA]##> ##[YODA]##>']

In [147]:
tokenizer_llama = AutoTokenizer.from_pretrained("meta-llama/Llama-2-7b-hf")
tokenizer_llama.pad_token = tokenizer_llama.unk_token
tokenizer_llama.pad_token_id = tokenizer_llama.unk_token_id

OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/meta-llama/Llama-2-7b.
401 Client Error. (Request ID: Root=1-69febf0b-0c1ae4284dc9e5bb30d8f74c;439a7d71-2c3e-49a7-9f00-7b2de729fd93)

Cannot access gated repo for url https://huggingface.co/meta-llama/Llama-2-7b/resolve/main/config.json.
Access to model meta-llama/Llama-2-7b is restricted. You must have access to it and be authenticated to access it. Please log in.